In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import optuna
import os
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 로드
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

# 2. 결측치 및 중복행 처리
train = train.drop_duplicates(subset=[col for col in train.columns if col != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

# 3. 파생변수 생성 (19개 확정본)
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)
    
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    
    return data

train = add_features(train)
test = add_features(test)

# 4. 상위 피처 이상치(Outlier) 처리 (IQR Clipping)
top_num_features = [
    'cholesterol', 'height', 'glucose', 'glucose_chol_ratio', 'bmi', 
    'weight', 'cardio_metabolic_load', 'map', 'pulse_pressure', 
    'systolic_blood_pressure', 'diastolic_blood_pressure', 'age'
]

for col in top_num_features:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    train[col] = train[col].clip(lower_bound, upper_bound)
    test[col] = test[col].clip(lower_bound, upper_bound)

# 5. 인코딩 (LightGBM Category dtype 활용)
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

cat_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'activity', 'edu_level']

for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

# 6. 타겟 변수 로그 변환
y_train_log = np.log1p(y_train)

# 7. Optuna를 활용한 하이퍼파라미터 튜닝
def objective(trial):
    params = {
        'objective': 'regression_l1', # MAE 최적화
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'verbose': -1,
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255, step=16),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_mae = []
    
    for tr_idx, val_idx in kf.split(x_train):
        X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
        y_tr, y_val = y_train_log.iloc[tr_idx], y_train_log.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, categorical_feature=cat_cols)
        
        pred_log = model.predict(X_val)
        pred = np.expm1(pred_log) # 역변환
        true = np.expm1(y_val)
        
        cv_mae.append(mean_absolute_error(true, pred))
        
    return np.mean(cv_mae)

print("★ Optuna 튜닝 시작 (30회 반복) ★")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

print(f"\n★ Best Optuna CV MAE: {study.best_value:.4f}")
print("Best Params:", study.best_params)

# 8. 최적 파라미터로 최종 학습 및 자체 점수 채점 (OOF)
best_params = study.best_params
best_params.update({
    'objective': 'regression_l1', 
    'metric': 'mae', 
    'random_state': 42, 
    'verbose': -1
})

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_log = np.zeros(len(x_train))
test_preds_log = np.zeros(len(x_test))

for tr_idx, val_idx in kf.split(x_train):
    X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
    y_tr, y_val = y_train_log.iloc[tr_idx], y_train_log.iloc[val_idx]
    
    final_model = lgb.LGBMRegressor(**best_params)
    final_model.fit(X_tr, y_tr, categorical_feature=cat_cols)
    
    oof_preds_log[val_idx] = final_model.predict(X_val)
    test_preds_log += final_model.predict(x_test) / kf.n_splits

# 자체 OOF 점수 계산 (역변환 후 원본 y_train과 비교)
oof_preds = np.expm1(oof_preds_log)
final_cv_mae = mean_absolute_error(y_train, oof_preds)
print(f"\n★ 최종 모델 자체 점수 (OOF CV MAE): {final_cv_mae:.4f}")

# 9. 예측값 역변환 및 클리핑
test_preds = np.expm1(test_preds_log)
test_preds = np.clip(test_preds, 0, 1)

# 10. 제출 파일 저장
os.makedirs('../submissions', exist_ok=True)
sample_submission['stress_score'] = test_preds
submit_path = '../submissions/submit_13_advanced_lgbm_optuna.csv'
sample_submission.to_csv(submit_path, index=False)
print(f"★ 제출 파일 생성 완료: {submit_path}")